# Behind the scenes: Day Three
### SDM dataset preparation

**Prerequisites:** run `day2_bts.ipynb` first. The following files must exist:
- `data/raw/GBIF/flying_fox.csv` — GBIF occurrence records for *Pteropus vampyrus*
- `data/raw/Beck/beck_philippines_1991_2020.nc` — 1 km monthly climatology

This notebook:
1. Cleans and filters the GBIF occurrence records
2. Generates random pseudo-absence points across the Philippines
3. Extracts Beck climate values (annual mean T, annual total P) at all points
4. Saves the SDM-ready dataset to `data/processed/sdm_flying_fox.csv`

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

from workshop_utils import RAW_DIR, PROCESSED_DIR
print('Ready.')

---
## 1. Load and clean GBIF occurrence records

In [ ]:
fox_raw = pd.read_csv(RAW_DIR / 'GBIF' / 'flying_fox.csv')
print(f'Raw records: {len(fox_raw)}')

# Keep only Philippines records with coordinates inside the valid bbox
fox = fox_raw[
    fox_raw['lon'].between(116.0, 127.0) &
    fox_raw['lat'].between(4.5, 21.5)
].copy()

# Remove spatial duplicates rounded to 0.01° (~1 km)
fox['lon_r'] = fox['lon'].round(2)
fox['lat_r'] = fox['lat'].round(2)
fox = fox.drop_duplicates(subset=['lon_r', 'lat_r']).drop(columns=['lon_r', 'lat_r'])

print(f'After filtering & deduplication: {len(fox)} presence points')
fox.head()

---
## 2. Generate pseudo-absence (background) points

We sample random points from the Philippines land area using the Beck KG
classification as a land mask (any non-zero KG code = land).

In [ ]:
ds_beck = xr.open_dataset(RAW_DIR / 'Beck' / 'beck_philippines_1991_2020.nc')

if 'kg' in ds_beck:
    land_mask = (ds_beck['kg'].values > 0)
    lats = ds_beck['latitude'].values
    lons = ds_beck['longitude'].values
    land_rows, land_cols = np.where(land_mask)
    print(f'Land pixels available: {len(land_rows):,}')
else:
    land_rows, land_cols, lats, lons = None, None, None, None
    print('WARNING: kg layer missing — will fall back to random bbox sampling')

n_pseudo = min(len(fox) * 10, 2000)   # up to 10× presence count, capped at 2000
rng = np.random.default_rng(42)

if land_rows is not None:
    idx = rng.choice(len(land_rows), size=n_pseudo, replace=False)
    pseudo_df = pd.DataFrame({
        'lon': lons[land_cols[idx]],
        'lat': lats[land_rows[idx]],
    })
else:
    pseudo_df = pd.DataFrame({
        'lon': rng.uniform(116.0, 127.0, n_pseudo),
        'lat': rng.uniform(4.5, 21.5, n_pseudo),
    })

print(f'Generated {len(pseudo_df)} pseudo-absence points')

---
## 3. Extract climate predictors at all points

We derive two bioclimatic variables from the Beck monthly data:
- **bio1** — annual mean temperature (mean of the 12 monthly Tavg grids, °C)
- **bio12** — annual precipitation (sum of the 12 monthly Prec grids, mm/yr)

In [ ]:
# Compute annual summaries
tavg_vars = [v for v in ds_beck if v.startswith('tavg_')]
prec_vars  = [v for v in ds_beck if v.startswith('prec_')]

bio1  = ds_beck[tavg_vars].to_array(dim='month').mean('month')  if tavg_vars else None
bio12 = ds_beck[prec_vars].to_array(dim='month').sum('month')   if prec_vars else None

print(f'bio1  (annual mean T):  {"available" if bio1  is not None else "MISSING"}')
print(f'bio12 (annual precip):  {"available" if bio12 is not None else "MISSING"}')


def extract_values(df, da, var_name):
    """Nearest-neighbour extraction of a 2-D DataArray at lon/lat points."""
    vals = []
    for _, row in df.iterrows():
        try:
            v = float(da.sel(longitude=row['lon'], latitude=row['lat'], method='nearest'))
        except Exception:
            v = np.nan
        vals.append(v)
    return vals


print('\nExtracting values at presence points ...')
fox_out = fox[['lon', 'lat']].copy()
if bio1  is not None: fox_out['bio1']  = extract_values(fox_out, bio1,  'bio1')
if bio12 is not None: fox_out['bio12'] = extract_values(fox_out, bio12, 'bio12')
fox_out['presence'] = 1

print('Extracting values at pseudo-absence points ...')
if bio1  is not None: pseudo_df['bio1']  = extract_values(pseudo_df, bio1,  'bio1')
if bio12 is not None: pseudo_df['bio12'] = extract_values(pseudo_df, bio12, 'bio12')
pseudo_df['presence'] = 0

sdm_df = pd.concat([fox_out, pseudo_df], ignore_index=True).dropna()
n_pres = sdm_df['presence'].sum()
n_abs  = (sdm_df['presence'] == 0).sum()
print(f'\nSDM dataset: {n_pres} presences + {n_abs} background points')
sdm_df.describe()

---
## 4. Save processed dataset

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
out_path = PROCESSED_DIR / 'sdm_flying_fox.csv'
sdm_df.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
sdm_df.head(10)